# 00 — Dataset Exploration

Anime Sketch Colorization Pair (Kaggle, ~17,769 images).

- Inspect raw 1024x512 side-by-side format (**left = color target, right = sketch**)
- Verify the sketch/color split done by `AnimeColorizationDataset`
- Check image statistics, train/val/test split sizes
- Visualize sample pairs (paired and unpaired mode)

Split mapping: Kaggle `train/` → train; Kaggle `val/` sorted by filename, first half → val, second half → test.

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from src.data import AnimeColorizationDataset
from src.utils import seed_everything

seed_everything(42)

DATA_ROOT = Path("../data/anime_colorization")

## 1. Raw file format

One raw file straight from disk: a single 1024×512 RGB image holding both halves side by side.

In [ ]:
raw_files = sorted((DATA_ROOT / "train").glob("*.png"))

raw = Image.open(raw_files[1]).convert("RGB")
print("raw size (W, H):", raw.size, "| mode:", raw.mode)

fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(raw)
ax.axvline(raw.size[0] // 2, color="red", linestyle="--")
ax.text(0.25, 1.02, "left = COLOR", transform=ax.transAxes, ha="center", color="red")
ax.text(0.75, 1.02, "right = SKETCH", transform=ax.transAxes, ha="center", color="red")
ax.axis("off")
plt.show()

## 2. Splits and tensor format

`AnimeColorizationDataset` returns `{"sketch": Tensor[3,256,256], "color": Tensor[3,256,256]}` normalized to [-1, 1].

In [ ]:
datasets = {split: AnimeColorizationDataset(DATA_ROOT, split=split)
            for split in ("train", "val", "test")}
total = sum(len(ds) for ds in datasets.values())
for split, ds in datasets.items():
    print(f"{split:5s}: {len(ds):6d} images ({len(ds) / total:.1%})")

sample = datasets["train"][0]
print("\nsketch:", tuple(sample["sketch"].shape), sample["sketch"].dtype)
print("color :", tuple(sample["color"].shape), sample["color"].dtype)
print(f"value range: [{sample['sketch'].min():.3f}, {sample['sketch'].max():.3f}]")

val_files = {f.name for f in datasets["val"].files}
test_files = {f.name for f in datasets["test"].files}
print("val/test overlap:", len(val_files & test_files))

## 2. Sample pairs (paired mode)

Random train samples as returned by the dataset: top row sketch, bottom row the aligned color target.

In [ ]:
def denorm(t: torch.Tensor) -> np.ndarray:
    """[-1, 1] CHW tensor -> [0, 1] HWC array for imshow."""
    return (t.permute(1, 2, 0).numpy() + 1) / 2

n = 4
indices = torch.randint(len(datasets["train"]), (n,))
fig, axes = plt.subplots(2, n, figsize=(3 * n, 6.5))
for col, idx in enumerate(indices):
    sample = datasets["train"][int(idx)]
    axes[0, col].imshow(denorm(sample["sketch"]))
    axes[0, col].set_title(f"sketch #{int(idx)}")
    axes[1, col].imshow(denorm(sample["color"]))
    axes[1, col].set_title(f"color #{int(idx)}")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Unpaired mode (CycleGAN setting)

With `paired=False` the color target at index `i` comes from a fixed seeded permutation of the indices, so sketches and colors are decoupled.


In [ ]:
unpaired = AnimeColorizationDataset(DATA_ROOT, split="train", paired=False)

n = 4
idxs = [8935, 1424, 9674, 6912]

fig, axes = plt.subplots(2, n, figsize=(3 * n, 6.5))
for col in range(n):
    idx = idxs[col]
    sample = unpaired[idx]

    axes[0, col].imshow(denorm(sample["sketch"]))
    axes[0, col].set_title(f"sketch #{col}")
    axes[1, col].imshow(denorm(sample["color"]))
    axes[1, col].set_title(f"color (shuffled)")
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("paired=False — colors are decoupled from sketches", y=1.0)
plt.tight_layout()
plt.show()